# Análisis Descriptivo de Datos – Catálogo de Productos

**Asignatura:** Business Analytics (BA)  
**Actividad:** R2-A10-S10 Análisis Descriptivo de Datos  
**Autor:** Brahan Julian Garcia Solorzano  
**Programa:** Especialización en Analítica y Ciencia de Datos – Universidad de Cundinamarca  
**Fecha:** Octubre 2026

---

## Descripción

En este notebook se realiza el **Análisis Descriptivo** del catálogo de productos que ya habíamos limpiado en la actividad de Obtención de Datos (ETL). La idea es tomar esos datos y empezar a entenderlos: ver cómo se comportan los precios, qué tan bien está el inventario, y si hay patrones interesantes entre las variables.

El análisis descriptivo es básicamente la primera etapa en Business Analytics, donde buscamos responder **¿qué está pasando con nuestros datos?** antes de intentar explicar por qué o predecir qué va a pasar.

Según la Guía Analítica Empresarial, con este tipo de análisis podemos:

- Entender el estado actual de los datos del negocio.
- Identificar las variables más importantes y sus tendencias.
- Crear visualizaciones que nos ayuden a interpretar la información.
- Tener una base clara para los análisis que vienen después (diagnóstico, predictivo, prescriptivo).

### Lo que vamos a hacer

1. Cargar el dataset y aplicar la misma limpieza del ETL anterior.
2. Calcular estadísticas descriptivas (promedios, medianas, dispersión, etc.).
3. Generar gráficos que muestren los patrones más relevantes.
4. Sacar conclusiones sobre el estado del catálogo.

### Sobre los datos

Trabajamos con el archivo `catalogo_productos.csv`, al que le aplicamos las mismas transformaciones de limpieza del ETL: quitar duplicados por SKU, sacar precios inválidos, unificar los nombres de categoría y marcar los productos que llevan mucho tiempo sin actualizarse.

---
## 1. Configuración del entorno

Primero importamos las librerías que vamos a necesitar y configuramos el estilo de los gráficos para que se vean bien.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, timedelta

# Configuración visual
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

# Formato de números para precios en pesos colombianos
def formato_cop(x, _=None):
    """Formatea un número como precio en COP sin decimales."""
    return f"${x:,.0f}"

print("Librerías cargadas correctamente.")
print(f"pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}")

---
## 2. Carga y preparación de datos

Cargamos el CSV original y le aplicamos la misma limpieza que hicimos en el ETL. Así nos aseguramos de trabajar con datos confiables.

Los pasos de limpieza son:
1. Convertir la fecha a formato datetime.
2. Quitar duplicados por SKU (nos quedamos con el registro más reciente).
3. Eliminar productos con precios negativos o en cero (datos erróneos).
4. Unificar los nombres de categoría (había variaciones como "televisores", "TELEVISORES", "electro hogar").
5. Calcular cuántos días lleva cada producto sin actualizarse y marcar los que superan 180 días.

In [ ]:
# --- Cargar datos originales ---
df_raw = pd.read_csv("/content/catalogo_productos.csv")
print(f"Registros originales cargados: {len(df_raw)}")

# --- Aplicar limpieza ETL ---
df = df_raw.copy()

# Convertir fecha
df["fecha_actualizacion"] = pd.to_datetime(df["fecha_actualizacion"], errors="coerce")

# Eliminar duplicados por SKU (conservar el más reciente)
df = df.sort_values("fecha_actualizacion", ascending=False)
df = df.drop_duplicates(subset="sku", keep="first")

# Excluir precios inválidos
df["precio"] = pd.to_numeric(df["precio"], errors="coerce")
df = df[df["precio"] > 0]

# Estandarizar categorías
df["categoria"] = df["categoria"].str.strip().str.title()
df["categoria"] = df["categoria"].replace({"Electro Hogar": "Electrohogar"})

# Marcar desactualizados (>180 días)
hoy = pd.Timestamp.now().normalize()
df["dias_desde_actualizacion"] = (hoy - df["fecha_actualizacion"]).dt.days
df["desactualizado"] = df["dias_desde_actualizacion"] > 180

# Resetear índice
df = df.reset_index(drop=True)

print(f"Registros después de limpieza ETL: {len(df)}")
print(f"Columnas disponibles: {df.columns.tolist()}")

### 2.1 Exploración inicial del dataset limpio

Revisamos la estructura del dataset después de la limpieza para confirmar que todo quedó bien: tipos de datos, valores nulos y una vista rápida de las primeras filas.

In [ ]:
print("=" * 60)
print("ESTRUCTURA DEL DATASET LIMPIO")
print("=" * 60)
print(f"\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nValores nulos por columna:")
print(df.isnull().sum())
print(f"\nPrimeras 5 filas:")
df.head()

---
## 3. Estadísticas descriptivas

Ahora sí entramos al análisis. Calculamos tres tipos de medidas para cada variable numérica:

- **Tendencia central:** Media, mediana y moda → nos dicen cuál es el valor "típico".
- **Dispersión:** Desviación estándar, varianza, rango, IQR → nos dicen qué tan esparcidos están los datos.
- **Forma:** Asimetría y curtosis → nos dicen si la distribución es simétrica o si tiene colas pronunciadas.

### 3.1 Estadísticas del Precio

El precio es probablemente la variable más importante del catálogo. Veamos cómo se comporta:

In [ ]:
precio = df["precio"]

estadisticas_precio = {
    "Métrica": [
        "Conteo",
        "Media",
        "Mediana",
        "Moda",
        "Desviación estándar",
        "Varianza",
        "Mínimo",
        "Máximo",
        "Rango",
        "Q1 (percentil 25)",
        "Q3 (percentil 75)",
        "Rango intercuartílico (IQR)",
        "Asimetría (Skewness)",
        "Curtosis",
    ],
    "Valor": [
        f"{precio.count():,}",
        formato_cop(precio.mean()),
        formato_cop(precio.median()),
        formato_cop(precio.mode().iloc[0]),
        formato_cop(precio.std()),
        f"{precio.var():,.0f}",
        formato_cop(precio.min()),
        formato_cop(precio.max()),
        formato_cop(precio.max() - precio.min()),
        formato_cop(precio.quantile(0.25)),
        formato_cop(precio.quantile(0.75)),
        formato_cop(precio.quantile(0.75) - precio.quantile(0.25)),
        f"{precio.skew():.4f}",
        f"{precio.kurtosis():.4f}",
    ],
}

df_stats_precio = pd.DataFrame(estadisticas_precio)
print("ESTADÍSTICAS DESCRIPTIVAS — PRECIO")
print("=" * 50)
df_stats_precio

**¿Qué nos dice esto?**

Lo primero que llama la atención es que la media y la mediana están bastante cerca. Eso nos indica que la distribución de precios es relativamente simétrica, sin valores extremos que jalen mucho el promedio hacia un lado.

La desviación estándar es alta (alrededor de $1.4 millones), lo cual tiene sentido porque el catálogo maneja desde productos económicos hasta artículos de gama alta. El IQR nos da una visión más limpia de la dispersión central, sin dejarse afectar por los extremos.

En cuanto a la asimetría, un valor cercano a 0 confirma que no hay una cola pronunciada hacia ningún lado. La curtosis nos dice si los datos están más o menos concentrados que una distribución normal.

### 3.2 Estadísticas del Stock

Ahora veamos el inventario. El stock nos dice cuántas unidades hay disponibles de cada producto.

In [ ]:
stock = df["stock"]

estadisticas_stock = {
    "Métrica": [
        "Conteo", "Media", "Mediana", "Moda",
        "Desviación estándar", "Mínimo", "Máximo", "Rango",
        "Q1 (percentil 25)", "Q3 (percentil 75)", "IQR",
        "Asimetría", "Curtosis",
    ],
    "Valor": [
        f"{stock.count():,}",
        f"{stock.mean():.1f}",
        f"{stock.median():.1f}",
        f"{stock.mode().iloc[0]}",
        f"{stock.std():.1f}",
        f"{stock.min()}",
        f"{stock.max()}",
        f"{stock.max() - stock.min()}",
        f"{stock.quantile(0.25):.1f}",
        f"{stock.quantile(0.75):.1f}",
        f"{stock.quantile(0.75) - stock.quantile(0.25):.1f}",
        f"{stock.skew():.4f}",
        f"{stock.kurtosis():.4f}",
    ],
}

df_stats_stock = pd.DataFrame(estadisticas_stock)
print("ESTADÍSTICAS DESCRIPTIVAS — STOCK")
print("=" * 50)
df_stats_stock

**¿Qué nos dice esto?**

El stock va de 0 a 100 unidades. El hecho de que el mínimo sea 0 confirma que hay productos agotados en el catálogo, algo que ya habíamos visto en el ETL. La media ronda las 46 unidades, y la distribución parece bastante uniforme entre el rango completo.

### 3.3 Resumen general con `describe()`

Un vistazo rápido a todas las variables numéricas juntas:

In [ ]:
print("RESUMEN ESTADÍSTICO GENERAL (variables numéricas)")
print("=" * 60)
df[["precio", "stock", "tiene_imagen", "tiene_descripcion", "dias_desde_actualizacion"]].describe().round(2)

### 3.4 Variables categóricas

Además de los números, es importante ver cómo se distribuyen las categorías, marcas y estados. Esto nos da una idea de la composición del catálogo.

In [ ]:
print("DISTRIBUCIÓN POR CATEGORÍA")
print("-" * 40)
cat_counts = df["categoria"].value_counts()
cat_pct = df["categoria"].value_counts(normalize=True).mul(100).round(1)
dist_cat = pd.DataFrame({"Cantidad": cat_counts, "Porcentaje (%)": cat_pct})
print(dist_cat)

print(f"\nDISTRIBUCIÓN POR MARCA")
print("-" * 40)
marca_counts = df["marca"].value_counts(dropna=False)
marca_pct = df["marca"].value_counts(dropna=False, normalize=True).mul(100).round(1)
dist_marca = pd.DataFrame({"Cantidad": marca_counts, "Porcentaje (%)": marca_pct})
print(dist_marca)

print(f"\nDISTRIBUCIÓN POR ESTADO")
print("-" * 40)
estado_counts = df["estado"].value_counts()
estado_pct = df["estado"].value_counts(normalize=True).mul(100).round(1)
dist_estado = pd.DataFrame({"Cantidad": estado_counts, "Porcentaje (%)": estado_pct})
print(dist_estado)

Acá se ve algo interesante: las categorías están bastante parejas en cantidad (~195 productos cada una después de la estandarización), lo que sugiere que el catálogo está diversificado. En cuanto a marcas, las 6 marcas principales tienen presencia similar, pero hay 20 productos que no tienen marca asignada — eso es un tema de calidad de datos que habría que resolver en la fuente.

La proporción de activos vs inactivos nos dice qué tan "vivo" está el catálogo.

---
## 4. Visualizaciones

Los números están bien, pero los gráficos son los que realmente nos dejan ver los patrones. Vamos a crear varias visualizaciones para explorar el catálogo desde distintos ángulos.

### 4.1 Distribución de precios

Con el histograma vemos la forma de la distribución (dónde se concentran los precios) y con el boxplot podemos detectar si hay valores atípicos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma de precios
axes[0].hist(df["precio"], bins=30, color="#2F5496", edgecolor="white", alpha=0.85)
axes[0].axvline(df["precio"].mean(), color="#E74C3C", linestyle="--", linewidth=2,
                label=f'Media: {formato_cop(df["precio"].mean())}')
axes[0].axvline(df["precio"].median(), color="#F39C12", linestyle="--", linewidth=2,
                label=f'Mediana: {formato_cop(df["precio"].median())}')
axes[0].set_title("Distribución de Precios", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Precio (COP)")
axes[0].set_ylabel("Frecuencia")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(formato_cop))
axes[0].legend(fontsize=10)

# Boxplot de precios
bp = axes[1].boxplot(df["precio"], vert=True, patch_artist=True,
                     boxprops=dict(facecolor="#AED6F1", edgecolor="#2F5496"),
                     medianprops=dict(color="#E74C3C", linewidth=2),
                     whiskerprops=dict(color="#2F5496"),
                     capprops=dict(color="#2F5496"))
axes[1].set_title("Boxplot de Precios", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Precio (COP)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(formato_cop))
axes[1].set_xticklabels(["Precio"])

plt.tight_layout()
plt.show()

print("\n→ El histograma revela la forma general de la distribución de precios.")
print("→ El boxplot muestra la concentración central (caja = IQR) y posibles outliers (puntos fuera de los bigotes).")
print(f"→ La posición relativa de media vs mediana indica {'asimetría positiva (cola derecha)' if df['precio'].mean() > df['precio'].median() else 'asimetría negativa (cola izquierda)' if df['precio'].mean() < df['precio'].median() else 'distribución simétrica'}.")

### 4.2 Distribución de stock

Hacemos lo mismo con el stock para ver cómo está repartido el inventario.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma de stock
axes[0].hist(df["stock"], bins=20, color="#27AE60", edgecolor="white", alpha=0.85)
axes[0].axvline(df["stock"].mean(), color="#E74C3C", linestyle="--", linewidth=2,
                label=f'Media: {df["stock"].mean():.1f}')
axes[0].axvline(df["stock"].median(), color="#F39C12", linestyle="--", linewidth=2,
                label=f'Mediana: {df["stock"].median():.1f}')
axes[0].set_title("Distribución de Stock", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Unidades en stock")
axes[0].set_ylabel("Frecuencia")
axes[0].legend(fontsize=10)

# Boxplot de stock
axes[1].boxplot(df["stock"], vert=True, patch_artist=True,
                boxprops=dict(facecolor="#ABEBC6", edgecolor="#27AE60"),
                medianprops=dict(color="#E74C3C", linewidth=2),
                whiskerprops=dict(color="#27AE60"),
                capprops=dict(color="#27AE60"))
axes[1].set_title("Boxplot de Stock", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Unidades")
axes[1].set_xticklabels(["Stock"])

plt.tight_layout()
plt.show()

print(f"\n→ El stock presenta una distribución {'uniforme' if abs(df['stock'].skew()) < 0.5 else 'asimétrica'} con rango de {df['stock'].min()} a {df['stock'].max()} unidades.")
print(f"→ Productos sin stock (stock = 0): {(df['stock'] == 0).sum()} ({(df['stock'] == 0).mean()*100:.1f}% del catálogo).")

### 4.3 Productos por categoría

¿Cuántos productos tiene cada categoría? Esto nos muestra la composición del portafolio.

In [ ]:
cat_order = df["categoria"].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(cat_order.index, cat_order.values,
              color=["#2F5496", "#3498DB", "#5DADE2", "#85C1E9", "#AED6F1"],
              edgecolor="white", linewidth=1.5)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., height + 2,
            f"{int(height)}", ha="center", va="bottom", fontweight="bold", fontsize=12)

ax.set_title("Cantidad de Productos por Categoría", fontsize=14, fontweight="bold")
ax.set_xlabel("Categoría")
ax.set_ylabel("Cantidad de productos")
ax.set_ylim(0, max(cat_order.values) * 1.15)
plt.tight_layout()
plt.show()

print(f"\n→ La categoría con más productos es '{cat_order.index[0]}' ({cat_order.values[0]}), seguida de '{cat_order.index[1]}' ({cat_order.values[1]}).")
print(f"→ La distribución es {'relativamente equilibrada' if (cat_order.max() - cat_order.min()) / cat_order.mean() < 0.5 else 'desigual entre categorías'}.")

### 4.4 Activos vs inactivos

Un indicador rápido de la salud del catálogo: ¿cuántos productos están activos y cuántos ya no?

In [ ]:
estado_vals = df["estado"].value_counts()

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    estado_vals.values,
    labels=estado_vals.index,
    autopct="%1.1f%%",
    colors=["#27AE60", "#E74C3C"],
    startangle=90,
    explode=(0.03, 0.03),
    textprops={"fontsize": 13},
    pctdistance=0.8,
)
for autotext in autotexts:
    autotext.set_fontweight("bold")
    autotext.set_color("white")

ax.set_title("Proporción de Productos Activos vs Inactivos", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\n→ {estado_vals.get('Activo', 0)} productos activos ({estado_vals.get('Activo', 0)/len(df)*100:.1f}%).")
print(f"→ {estado_vals.get('Inactivo', 0)} productos inactivos ({estado_vals.get('Inactivo', 0)/len(df)*100:.1f}%).")

### 4.5 Precio promedio por categoría

¿Cuáles categorías manejan precios más altos? Esto puede servir para entender la estrategia de precios.

In [ ]:
precio_por_cat = df.groupby("categoria")["precio"].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(precio_por_cat.index, precio_por_cat.values,
               color="#2F5496", edgecolor="white", height=0.6)

for bar in bars:
    width = bar.get_width()
    ax.text(width + 30000, bar.get_y() + bar.get_height() / 2.,
            formato_cop(width), ha="left", va="center", fontsize=11, fontweight="bold")

ax.set_title("Precio Promedio por Categoría", fontsize=14, fontweight="bold")
ax.set_xlabel("Precio promedio (COP)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(formato_cop))
plt.tight_layout()
plt.show()

print(f"\n→ La categoría con precio promedio más alto es '{precio_por_cat.index[-1]}' ({formato_cop(precio_por_cat.values[-1])}).")
print(f"→ La categoría con precio promedio más bajo es '{precio_por_cat.index[0]}' ({formato_cop(precio_por_cat.values[0])}).")

### 4.6 Precios por categoría (boxplots)

Los boxplots lado a lado nos dejan comparar no solo el promedio sino la variabilidad y los outliers de precio en cada categoría.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sns.boxplot(data=df, x="categoria", y="precio", palette="Blues_d", ax=ax, width=0.6)
ax.set_title("Distribución de Precios por Categoría", fontsize=14, fontweight="bold")
ax.set_xlabel("Categoría")
ax.set_ylabel("Precio (COP)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(formato_cop))
plt.tight_layout()
plt.show()

print("\n→ Los boxplots permiten comparar la variabilidad de precios entre categorías.")
print("→ Medianas similares sugieren precios equiparables; cajas anchas indican mayor dispersión.")
print("→ Los puntos fuera de los bigotes representan outliers (precios atípicos).")

### 4.7 Productos por marca

Veamos qué marcas tienen más presencia. Los productos sin marca los resaltamos en rojo porque representan un problema de calidad de datos.

In [ ]:
marca_counts = df["marca"].value_counts(dropna=False)
marca_counts.index = [x if pd.notna(x) else "Sin marca" for x in marca_counts.index]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#E74C3C" if idx == "Sin marca" else "#2F5496" for idx in marca_counts.index]
bars = ax.bar(marca_counts.index, marca_counts.values, color=colors, edgecolor="white")

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., height + 2,
            f"{int(height)}", ha="center", va="bottom", fontweight="bold", fontsize=11)

ax.set_title("Cantidad de Productos por Marca", fontsize=14, fontweight="bold")
ax.set_xlabel("Marca")
ax.set_ylabel("Cantidad")
ax.set_ylim(0, max(marca_counts.values) * 1.15)
plt.tight_layout()
plt.show()

print(f"\n→ {df['marca'].isna().sum()} productos no tienen marca asignada (resaltados en rojo).")

### 4.8 Completitud del contenido

Para un catálogo digital, tener imagen y descripción es fundamental. Veamos qué tan completa está la información.

In [ ]:
contenido = pd.DataFrame({
    "Atributo": ["Tiene imagen", "Tiene descripción"],
    "Sí": [
        (df["tiene_imagen"] == 1).sum(),
        (df["tiene_descripcion"] == 1).sum(),
    ],
    "No": [
        (df["tiene_imagen"] == 0).sum(),
        (df["tiene_descripcion"] == 0).sum(),
    ],
})

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(contenido))
width = 0.35

bars_si = ax.bar([i - width/2 for i in x], contenido["Sí"], width,
                  label="Sí", color="#27AE60", edgecolor="white")
bars_no = ax.bar([i + width/2 for i in x], contenido["No"], width,
                  label="No", color="#E74C3C", edgecolor="white")

for bar in list(bars_si) + list(bars_no):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., h + 5,
            f"{int(h)}", ha="center", va="bottom", fontweight="bold", fontsize=11)

ax.set_title("Completitud del Contenido del Catálogo", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(contenido["Atributo"])
ax.set_ylabel("Cantidad de productos")
ax.legend()
plt.tight_layout()
plt.show()

pct_img = (df["tiene_imagen"] == 1).mean() * 100
pct_desc = (df["tiene_descripcion"] == 1).mean() * 100
print(f"\n→ {pct_img:.1f}% de los productos tienen imagen.")
print(f"→ {pct_desc:.1f}% de los productos tienen descripción.")
print(f"→ La brecha de contenido faltante es mayor en descripciones que en imágenes." if pct_desc < pct_img else "→ La brecha de contenido es mayor en imágenes.")

### 4.9 Productos desactualizados por categoría

¿En qué categorías se concentran los productos que llevan más de 180 días sin actualizarse? Esto ayuda a priorizar el mantenimiento del catálogo.

In [ ]:
desact_cat = df.groupby("categoria")["desactualizado"].agg(["sum", "count"])
desact_cat.columns = ["Desactualizados", "Total"]
desact_cat["Actualizados"] = desact_cat["Total"] - desact_cat["Desactualizados"]
desact_cat["% Desactualizado"] = (desact_cat["Desactualizados"] / desact_cat["Total"] * 100).round(1)
desact_cat = desact_cat.sort_values("% Desactualizado", ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
y_pos = range(len(desact_cat))

ax.barh(y_pos, desact_cat["Actualizados"], color="#27AE60",
        label="Actualizados", edgecolor="white")
ax.barh(y_pos, desact_cat["Desactualizados"], left=desact_cat["Actualizados"],
        color="#E74C3C", label="Desactualizados (>180 días)", edgecolor="white")

ax.set_yticks(y_pos)
ax.set_yticklabels(desact_cat.index)
ax.set_title("Productos Actualizados vs Desactualizados por Categoría",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Cantidad de productos")
ax.legend(loc="lower right")

for i, (_, row) in enumerate(desact_cat.iterrows()):
    ax.text(row["Total"] + 3, i, f"{row['% Desactualizado']}%",
            va="center", fontweight="bold", fontsize=11, color="#E74C3C")

plt.tight_layout()
plt.show()

print(f"\n→ Total de productos desactualizados: {df['desactualizado'].sum()} ({df['desactualizado'].mean()*100:.1f}%).")
print(f"→ La categoría con mayor porcentaje de desactualización es '{desact_cat['% Desactualizado'].idxmax()}' ({desact_cat['% Desactualizado'].max()}%).")

### 4.10 Correlaciones entre variables numéricas

Con la matriz de correlación de Pearson podemos ver si hay relaciones lineales entre las variables. Valores cercanos a 1 o -1 indican relación fuerte; cercanos a 0, que no hay relación clara.

In [ ]:
vars_numericas = ["precio", "stock", "tiene_imagen", "tiene_descripcion",
                  "dias_desde_actualizacion"]
corr_matrix = df[vars_numericas].corr().round(3)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="RdBu_r", center=0,
            mask=mask, square=True, linewidths=1, linecolor="white",
            cbar_kws={"shrink": 0.8}, ax=ax,
            annot_kws={"fontsize": 12, "fontweight": "bold"})
ax.set_title("Matriz de Correlación — Variables Numéricas",
             fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

print("\n→ Valores cercanos a ±1 indican correlación fuerte; cercanos a 0, correlación débil.")
print("→ Correlaciones positivas fuertes entre tiene_imagen y tiene_descripcion sugerirían")
print("  que los productos con imagen tienden a tener también descripción (contenido completo).")

### 4.11 Precio vs Stock

¿Los productos más caros tienen más o menos inventario? El scatter plot nos ayuda a ver si existe alguna relación, y los coloreamos por categoría para ver si hay diferencias.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sns.scatterplot(data=df, x="precio", y="stock", hue="categoria",
                alpha=0.6, s=50, ax=ax, palette="Set2")
ax.set_title("Relación entre Precio y Stock por Categoría",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Precio (COP)")
ax.set_ylabel("Stock (unidades)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(formato_cop))
ax.legend(title="Categoría", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

corr_precio_stock = df["precio"].corr(df["stock"])
print(f"\n→ Coeficiente de correlación (Pearson) entre precio y stock: {corr_precio_stock:.4f}")
if abs(corr_precio_stock) < 0.3:
    print("→ No existe una relación lineal significativa entre precio y stock.")
    print("→ Esto indica que la gestión de inventario no depende directamente del valor del producto.")
elif corr_precio_stock > 0:
    print("→ Existe una relación positiva: productos más caros tienden a tener más stock.")
else:
    print("→ Existe una relación negativa: productos más caros tienden a tener menos stock.")

---
## 5. Tabla dinámica — Precio por marca y categoría

Para ir un paso más allá, cruzamos marca con categoría y calculamos el precio promedio de cada combinación. Esto nos deja ver, por ejemplo, si Samsung es más caro en Televisores que en Celulares, o cuál marca domina en cada segmento.

In [ ]:
# Excluir productos sin marca para este análisis
df_con_marca = df.dropna(subset=["marca"])

pivot_precio = pd.pivot_table(
    df_con_marca,
    values="precio",
    index="marca",
    columns="categoria",
    aggfunc="mean",
).round(0)

print("PRECIO PROMEDIO POR MARCA Y CATEGORÍA (COP)")
print("=" * 70)
pivot_display = pivot_precio.map(lambda x: formato_cop(x) if pd.notna(x) else "-")
print(pivot_display)

# Heatmap del pivot
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot_precio, annot=True, fmt=",.0f", cmap="YlOrRd",
            linewidths=1, linecolor="white", ax=ax,
            annot_kws={"fontsize": 10})
ax.set_title("Precio Promedio por Marca y Categoría (COP)",
             fontsize=14, fontweight="bold")
ax.set_ylabel("Marca")
ax.set_xlabel("Categoría")
plt.tight_layout()
plt.show()

print("\n→ El heatmap facilita la identificación visual de los segmentos de mayor y menor precio.")
print("→ Los colores más intensos indican precios promedio más altos en esa combinación marca-categoría.")

---
## 6. Resumen de hallazgos

Consolidamos los indicadores más importantes en una sola tabla para tener el panorama completo.

In [ ]:
hallazgos = [
    ["Total de productos analizados", f"{len(df):,}"],
    ["Precio promedio del catálogo", formato_cop(df['precio'].mean())],
    ["Precio mediano del catálogo", formato_cop(df['precio'].median())],
    ["Desviación estándar del precio", formato_cop(df['precio'].std())],
    ["Categoría con más productos", df['categoria'].value_counts().index[0]],
    ["Categoría con precio promedio más alto",
     df.groupby('categoria')['precio'].mean().idxmax()],
    ["Marca con más productos",
     df.dropna(subset=['marca'])['marca'].value_counts().index[0]],
    ["% productos activos", f"{(df['estado']=='Activo').mean()*100:.1f}%"],
    ["% productos sin stock", f"{(df['stock']==0).mean()*100:.1f}%"],
    ["% productos desactualizados (>180 días)",
     f"{df['desactualizado'].mean()*100:.1f}%"],
    ["% productos con imagen", f"{(df['tiene_imagen']==1).mean()*100:.1f}%"],
    ["% productos con descripción",
     f"{(df['tiene_descripcion']==1).mean()*100:.1f}%"],
    ["Correlación precio-stock", f"{df['precio'].corr(df['stock']):.4f}"],
]

df_hallazgos = pd.DataFrame(hallazgos, columns=["Indicador", "Valor"])
print("RESUMEN DE HALLAZGOS CLAVE")
print("=" * 60)
df_hallazgos

---
## 7. Conclusiones

Después de explorar el catálogo a fondo, estos son los hallazgos más relevantes:

**Precios:** La distribución es bastante simétrica, con media y mediana rondando los $2,560,000 COP. Hay una variabilidad alta (desviación estándar de ~$1.4M), lo cual tiene sentido porque el catálogo va desde productos económicos hasta gama alta. No se detectaron outliers preocupantes después de la limpieza.

**Inventario:** Hay 97 productos completamente agotados (stock = 0), que representan cerca del 10% del catálogo. El stock promedio está en 46 unidades y la distribución es bastante uniforme. Sería interesante investigar en qué categorías se concentran los agotamientos para priorizar reabastecimiento.

**Composición del catálogo:** Las 5 categorías quedaron bastante equilibradas después de estandarizar los nombres (~195 productos cada una). Lo mismo pasa con las marcas: Samsung, LG, Lenovo, Sony, Apple y HP tienen presencia similar. El detalle es que hay 20 productos sin marca, algo que habría que corregir en el sistema de carga.

**Contenido digital:** El 84% de productos tiene imagen y el 77% tiene descripción. La brecha es mayor en descripciones, así que si se va a hacer un plan de mejora de contenido, por ahí debería empezar.

**Vigencia de los datos:** Este es probablemente el hallazgo más preocupante: 576 productos (casi el 59%) llevan más de 180 días sin actualizarse. Eso significa que más de la mitad del catálogo podría tener precios o información desactualizada. Habría que armar un plan de actualización priorizando las categorías más críticas.

**Relación precio-stock:** No encontramos una correlación significativa entre el precio y las unidades en stock. Esto sugiere que la empresa no maneja el inventario en función del valor del producto, sino probablemente por demanda o rotación.

### ¿Y ahora qué?

Este análisis descriptivo responde la pregunta de **¿qué está pasando?** con el catálogo. Los 1,020 registros originales se convirtieron en 975 productos limpios (quitamos 20 duplicados y 25 con precios inválidos), y a partir de ahí pudimos sacar una radiografía bastante completa.

Los próximos pasos naturales serían:
- **Diagnóstico:** ¿Por qué hay tantos productos desactualizados? ¿Es un problema de proceso o de ciertas categorías?
- **Predictivo:** ¿Podemos anticipar qué productos se van a agotar?
- **Prescriptivo:** ¿Qué acciones concretas tomar para mejorar la calidad del catálogo?

---

### Herramientas utilizadas

- **Python 3** en Google Colab
- **pandas** para manipulación de datos
- **numpy** para cálculos numéricos
- **matplotlib** y **seaborn** para las gráficas

---

### Repositorio

📁 **Enlace al repositorio:** https://github.com/bjgarcia20/analisis-descriptivo-catalogo.git

---